In [ ]:
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor, Lambda

from core.classes import CustomImageDataset



In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(100*100, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 2),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits



In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
model = NeuralNetwork().to(device)
print(model)

In [ ]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    loss_list = []
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)
        
        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        loss_list.append(loss)
    return loss_list

def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            #mask_correct = (pred.argmax(1) == y.argmax(1))
            correct += (pred.argmax(1) == y.argmax(1)).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return test_loss


In [ ]:
training_data = CustomImageDataset('train', transform=True, target_transform=Lambda(lambda y: torch.zeros(2, dtype=torch.float).scatter_(0, torch.tensor(y), value=1)))
test_data = CustomImageDataset('test', transform=True, target_transform=Lambda(lambda y: torch.zeros(2, dtype=torch.float).scatter_(0, torch.tensor(y), value=1)))
val_data = CustomImageDataset('val', transform=True, target_transform=Lambda(lambda y: torch.zeros(2, dtype=torch.float).scatter_(0, torch.tensor(y), value=1)))

train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)



In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
train_loss_list = []
test_loss_list = []
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer)
    train_loss_list.append(train_loss)
    test_loss = test_loop(test_dataloader, model, loss_fn)
    test_loss_list.append(test_loss)
print("Done!")

In [ ]:
torch.save(model, 'model_nn_1.pth')

In [ ]:
def plot_loss_and_validation_loss(train_loss_list, test_loss_list) -> None:
    list_loss_train = []
    list_loss_mean = []
    for tensor_list_epoch in train_loss_list:
        list_epoch = []
        for tensor_loss in tensor_list_epoch:
            list_loss_train.append(tensor_loss.item())
            list_epoch.append(tensor_loss.item())
        list_loss_mean.append(np.array(list_epoch).mean())

    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10,10))

    ax.plot(range(10*107), list_loss_train, color = 'green', label='Trainings loss')
    ax.plot(range(0, 10*107, 107), list_loss_mean, color = 'blue', label='Mean trainings loss per epoch')
    ax.plot(range(0, 10*107, 107), test_loss_list, color = 'orange', label='Test loss')
    ax.legend()
    ax.set_title('Trainings and test loss')

In [ ]:
plot_loss_and_validation_loss(train_loss_list, test_loss_list)